# A3b — Split aleatório 70-15-15 (2019–2025) e cenário sem 2020–2021

**Pessoa A (apoio ao A3).** Experimento **comparativo**: o projeto usa validação temporal como padrão; aqui medimos o que muda com um **split aleatório estratificado 70/15/15** e quando **2020 e 2021** (anos de pandemia) saem da base.

| Cenário | Anos (de `DT_INTERNA`) | Split |
|---|---|---|
| `todos_2019_2025` | 2019–2025 | aleatório 70/15/15, estratificado pelo alvo, seed 42 |
| `sem_2020_2021` | 2019, 2022–2025 | idem |

**Saídas** (em `data/analytical/`, `outputs/tables/pessoa_a/` e `outputs/figures/pessoa_a/`):
- `dataset_supervisionado_<cenário>.parquet` com a coluna **`split`** (`treino` / `validacao` / `holdout`) e `.meta.json` (versão, SHA-256, contagens).
- Tabelas de proporções, prevalência e anos por split; waterfall por cenário.
- **Gráficos (PNG)**: proporções 70/15/15, prevalência de óbito por split e composição dos splits por ano (`split_<cenário>.png`); comparação dos cenários (`split_comparacao_cenarios.png`).
- Benchmark rápido (regressão logística, treino → **validação**): split aleatório × temporal por ordem.

> O **holdout não é aberto** aqui (regra do projeto). O benchmark usa só treino e validação.
> Os dois datasets **não substituem** `dataset_supervisionado.parquet` (o do pipeline temporal do C2).

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa_a"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "minimas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
CENARIOS = None                     # None = usa configs/decisoes.yaml -> experimentos_split.cenarios
                                    # ou informe, ex.: {"todos_2019_2025": [2019, 2020, 2021, 2022, 2023, 2024, 2025]}
RODAR_BENCHMARK = True              # regressão logística treino -> validação (aleatório x temporal)
FIG_DIR = "outputs/figures/pessoa_a"  # onde salvar os gráficos (PNG)
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A3b")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

## 1. Cenários
Os anos vêm do `configs/decisoes.yaml` (`experimentos_split.cenarios`); a seed e as frações também.

In [ ]:
e = cfg["experimentos_split"]
CENARIOS = CENARIOS or {nome: spec["anos"] for nome, spec in e["cenarios"].items()}
print("Seed:", e["seed"], "| frações:", e["fracoes"], "| estratificado pelo alvo:", e["estratificar_por_alvo"])
for nome, anos in CENARIOS.items():
    print(f"  {nome}: {anos}")
anos_na_base = sorted(v.parse_date(df[cfg["cols"]["dt_interna"]]).dt.year.dropna().astype(int).unique())
print("Anos de DT_INTERNA presentes na base carregada:", anos_na_base)

## 2. Coorte, dataset e checagens por cenário
Waterfall do cenário → dataset com `split` → proporções 70/15/15, estratificação do alvo, anos em cada split, `NU_NOTIFIC` repetido entre splits e contrato do C2.

In [ ]:
metas = {}
for nome, anos in CENARIOS.items():
    print(f"\n===== {nome}: {anos} =====")
    metas[nome] = v.a3_split_aleatorio_cenario(df, cfg, rep, nome, anos, SUPERVISED_CONFIG)
    v.validar_contrato_c2(metas[nome]["arquivo"], SUPERVISED_CONFIG, rep)
    display(pd.read_csv(Path(OUT) / f"a1_waterfall_{nome}.csv"))
    display(pd.read_csv(Path(OUT) / f"a3_split_aleatorio_{nome}.csv"))
    display(pd.read_csv(Path(OUT) / f"a3_split_aleatorio_{nome}_por_ano.csv"))
    # (a) % de linhas x meta 70/15/15 | (b) prevalência de óbito por split | (c) composição dos splits em cada ano
    print("Figura:", v.grafico_split_cenario(rep, cfg, nome, anos, FIG_DIR, mostrar=True))

## 3. Benchmark: split aleatório × temporal (treino → validação)
Regressão logística balanceada, mesmas features do `supervised.yaml`. Se o aleatório der PR-AUC maior que o temporal, ele está **otimista** (registros do mesmo período nos dois lados): a decisão deve seguir o temporal.

In [ ]:
benchs = {}
if RODAR_BENCHMARK:
    for nome in CENARIOS:
        benchs[nome] = v.a3_benchmark_split(metas[nome]["arquivo"], SUPERVISED_CONFIG, cfg, rep, nome,
                                            max_rows=e["bench_max_linhas"], seed=e["seed"])
        display(benchs[nome])
else:
    print("Benchmark desligado (RODAR_BENCHMARK = False).")

## 4. Comparação entre os cenários

In [ ]:
comp = v.comparar_cenarios(metas, benchs, rep)
display(comp)
# N por ano (empilhado por split) nos dois cenários + prevalência de óbito por split
print("Figura:", v.grafico_comparacao_cenarios(rep, cfg, list(CENARIOS), FIG_DIR, mostrar=True))

## 5. Decisões

In [ ]:
E = "A3b"
v.log_decision(OUT, "D-13", "Experimento split aleatório 70/15/15 (2019–2025)",
               f"Split aleatório estratificado pelo alvo, seed {e['seed']}; dataset_supervisionado_todos_2019_2025", E,
               impacto="Comparativo; o padrão do projeto segue a validação temporal (README)",
               evidencia="a3_benchmark_split_*.csv")
v.log_decision(OUT, "D-14", "Cenário sem 2020 e 2021",
               "Mesma análise removendo 2020–2021 (pandemia); dataset_supervisionado_sem_2020_2021", E,
               impacto="Compara N, prevalência e desempenho com a base completa (comparacao_cenarios.csv)")
display(pd.read_csv(Path(OUT) / "tabela_decisoes.csv"))

## 6. Fechamento

In [ ]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

### Como o C pode usar a coluna `split`
O C2 atual divide por frações no tempo (`TemporalSplit`). Para usar o split aleatório, basta ler o dataset do cenário e filtrar pela coluna:

```python
df = pd.read_parquet(C.ANALYTICAL_DIR / "dataset_supervisionado_todos_2019_2025.parquet")
treino, validacao, holdout = (df[df["split"] == s] for s in ("treino", "validacao", "holdout"))
```

### Checklist
- [ ] Proporções 70/15/15 e prevalência de óbito iguais entre os splits (estratificação)
- [ ] Comparação aleatório × temporal interpretada (otimismo do aleatório)
- [ ] Comparação com e sem 2020–2021 registrada (N, prevalência, desempenho)
- [ ] Holdout **não** aberto